# 07 — Model-family screen and submissions

This notebook records lifecycle steps 8 and 9 for a bounded model-family and probability-combination screen. It keeps the existing 29-feature policy, frozen development partition and five predefined folds unchanged. Tree counts are selected inside each outer-training fold; the reserved local test is not reopened.

The screen compares every new standalone not only with the complete incumbent, but also separately with the incumbent Random Forest and histogram-boosting components. A bounded second wave uses the committed 81.555% XGBoost/Random Forest candidate as a stricter reference, requiring at least +0.05 percentage points plus the fold and class-recall safeguards. A second submission candidate must pass the original accuracy and class-recall gate and disagree with the first on at least 1% of out-of-fold labels.

In [1]:
from pathlib import Path
import json
import sys
import joblib
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
STAGE_DIR = PROJECT_DIR / 'stage-1-pump-it-up'
if not STAGE_DIR.is_dir():
    STAGE_DIR = PROJECT_DIR.parent if PROJECT_DIR.name == 'notebooks' else PROJECT_DIR
    PROJECT_DIR = STAGE_DIR.parent
SRC_DIR = STAGE_DIR / 'src'
RUNTIME_DIR = PROJECT_DIR / '.runtime' / 'gpu-model-screen'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from model_screen_submission import select_submission_candidate_names

screen = joblib.load(RUNTIME_DIR / 'combination-screen.joblib')
summary = screen['summary'].copy()
print(f"Recorded {len(summary)} standalone and probability-combination trials.")

Recorded 609 standalone and probability-combination trials.


In [2]:
display_columns = [
    'mean_accuracy', 'mean_gain', 'fold_wins', 'worst_fold_change',
    'repair_recall', 'non_functional_recall',
    'incumbent_disagreement', 'current_leader_gain',
    'current_leader_disagreement', 'passes_gate',
    'beats_current_leader_gate',
]
summary.loc[:, display_columns].head(25).style.format({
    'mean_accuracy': '{:.3%}',
    'mean_gain': '{:+.3%}',
    'worst_fold_change': '{:+.3%}',
    'repair_recall': '{:.3%}',
    'non_functional_recall': '{:.3%}',
    'incumbent_disagreement': '{:.3%}',
    'current_leader_gain': '{:+.3%}',
    'current_leader_disagreement': '{:.3%}',
})

,mean_accuracy,mean_gain,fold_wins,worst_fold_change,repair_recall,non_functional_recall,incumbent_disagreement,current_leader_gain,current_leader_disagreement,passes_gate,beats_current_leader_gate
trial,,,,,,,,,,,
55% XGBoost depth 8 child 1 [current one-hot] + 45% Random Forest,81.625%,+0.223%,4,-0.021%,34.859%,78.433%,2.109%,+0.069%,1.115%,True,True
60% equal XGBoost depth 8 child 1 seed bag + 40% Random Forest,81.618%,+0.217%,4,-0.042%,34.714%,78.323%,2.178%,+0.063%,0.909%,True,True
60% second-wave XGBoost variant bag + 40% Random Forest,81.597%,+0.196%,4,+0.000%,34.309%,78.307%,2.058%,+0.042%,0.627%,True,False
60% XGBoost depth 6/7/8 local bag + 40% Random Forest,81.595%,+0.194%,5,+0.021%,34.396%,78.318%,2.159%,+0.040%,0.640%,True,False
65% XGBoost depth 8 child 1 [current one-hot] + 35% Random Forest,81.589%,+0.187%,4,-0.032%,34.193%,78.208%,2.487%,+0.034%,1.138%,True,False
65% equal XGBoost depth 8 child 1 seed bag + 35% Random Forest,81.587%,+0.185%,5,+0.042%,34.251%,78.235%,2.395%,+0.032%,1.016%,True,False
65% second-wave XGBoost variant bag + 35% Random Forest,81.582%,+0.181%,5,+0.011%,33.961%,78.203%,2.361%,+0.027%,0.882%,True,False
55% equal XGBoost depth 8 child 1 seed bag + 45% Random Forest,81.578%,+0.177%,4,-0.147%,34.830%,78.389%,2.052%,+0.023%,1.056%,True,False
55% XGBoost depth 6/7/8 local bag + 45% Random Forest,81.570%,+0.168%,4,-0.042%,34.338%,78.427%,1.972%,+0.015%,0.619%,True,False


In [3]:
standalone_names = [
    name for name, recipe in screen['recipes'].items()
    if len(recipe) == 1 and recipe[0][0] == name
]
print('Standalone models and incumbent components:')
summary.loc[summary.index.intersection(standalone_names), display_columns].sort_values(
    'mean_accuracy', ascending=False
).style.format({
    'mean_accuracy': '{:.3%}', 'mean_gain': '{:+.3%}',
    'worst_fold_change': '{:+.3%}', 'repair_recall': '{:.3%}',
    'non_functional_recall': '{:.3%}', 'incumbent_disagreement': '{:.3%}',
    'current_leader_gain': '{:+.3%}', 'current_leader_disagreement': '{:.3%}',
})

Standalone models and incumbent components:


,mean_accuracy,mean_gain,fold_wins,worst_fold_change,repair_recall,non_functional_recall,incumbent_disagreement,current_leader_gain,current_leader_disagreement,passes_gate,beats_current_leader_gate
trial,,,,,,,,,,,
incumbent 50% Random Forest + 50% boosting,81.402%,+0.000%,0,+0.000%,33.643%,78.153%,0.000%,-0.154%,2.157%,False,False
XGBoost depth 8 child 1 [current one-hot] seed 20260822,81.019%,-0.383%,0,-0.610%,31.326%,77.299%,5.210%,-0.537%,4.099%,False,False
XGBoost depth 8 child 1 [current one-hot],81.014%,-0.387%,1,-0.621%,31.529%,77.102%,5.385%,-0.541%,4.249%,False,False
XGBoost depth 8 columns 0.75 [current one-hot],81.008%,-0.394%,0,-0.737%,31.674%,77.091%,5.301%,-0.547%,4.217%,False,False
XGBoost depth 8 child 1 [current one-hot] seed 20260823,80.955%,-0.446%,0,-0.758%,32.021%,77.102%,5.309%,-0.600%,4.169%,False,False
XGBoost lossguide 64 [current one-hot],80.947%,-0.455%,0,-0.800%,32.774%,77.156%,5.128%,-0.608%,4.295%,False,False
XGBoost depth 11 conservative [current one-hot],80.934%,-0.467%,0,-0.768%,31.442%,77.124%,5.107%,-0.621%,4.127%,False,False
XGBoost depth 8 rows 0.80 [current one-hot],80.928%,-0.473%,0,-0.684%,31.819%,77.036%,5.438%,-0.627%,4.346%,False,False
XGBoost depth 8 [current one-hot] seed 20260822,80.924%,-0.478%,0,-0.673%,31.124%,76.998%,5.455%,-0.631%,4.407%,False,False


In [4]:
component_pair_names = [
    name for name, recipe in screen['recipes'].items()
    if len(recipe) == 2
    and any(component in {'Random Forest', 'histogram boosting'} for component, _ in recipe)
    and name != screen['incumbent_name']
]
print('Strongest direct pairings with an incumbent component:')
summary.loc[summary.index.intersection(component_pair_names), display_columns].sort_values(
    'mean_accuracy', ascending=False
).head(20).style.format({
    'mean_accuracy': '{:.3%}', 'mean_gain': '{:+.3%}',
    'worst_fold_change': '{:+.3%}', 'repair_recall': '{:.3%}',
    'non_functional_recall': '{:.3%}', 'incumbent_disagreement': '{:.3%}',
    'current_leader_gain': '{:+.3%}', 'current_leader_disagreement': '{:.3%}',
})

Strongest direct pairings with an incumbent component:


,mean_accuracy,mean_gain,fold_wins,worst_fold_change,repair_recall,non_functional_recall,incumbent_disagreement,current_leader_gain,current_leader_disagreement,passes_gate,beats_current_leader_gate
trial,,,,,,,,,,,
55% XGBoost depth 8 child 1 [current one-hot] + 45% Random Forest,81.625%,+0.223%,4,-0.021%,34.859%,78.433%,2.109%,+0.069%,1.115%,True,True
60% equal XGBoost depth 8 child 1 seed bag + 40% Random Forest,81.618%,+0.217%,4,-0.042%,34.714%,78.323%,2.178%,+0.063%,0.909%,True,True
60% second-wave XGBoost variant bag + 40% Random Forest,81.597%,+0.196%,4,+0.000%,34.309%,78.307%,2.058%,+0.042%,0.627%,True,False
60% XGBoost depth 6/7/8 local bag + 40% Random Forest,81.595%,+0.194%,5,+0.021%,34.396%,78.318%,2.159%,+0.040%,0.640%,True,False
65% XGBoost depth 8 child 1 [current one-hot] + 35% Random Forest,81.589%,+0.187%,4,-0.032%,34.193%,78.208%,2.487%,+0.034%,1.138%,True,False
65% equal XGBoost depth 8 child 1 seed bag + 35% Random Forest,81.587%,+0.185%,5,+0.042%,34.251%,78.235%,2.395%,+0.032%,1.016%,True,False
65% second-wave XGBoost variant bag + 35% Random Forest,81.582%,+0.181%,5,+0.011%,33.961%,78.203%,2.361%,+0.027%,0.882%,True,False
55% equal XGBoost depth 8 child 1 seed bag + 45% Random Forest,81.578%,+0.177%,4,-0.147%,34.830%,78.389%,2.052%,+0.023%,1.056%,True,False
55% XGBoost depth 6/7/8 local bag + 45% Random Forest,81.570%,+0.168%,4,-0.042%,34.338%,78.427%,1.972%,+0.015%,0.619%,True,False


In [5]:
selected_names = select_submission_candidate_names(screen)
passing = summary.loc[summary['passes_gate'], display_columns]
print(f"Gate-passers: {len(passing)}")
print('Selected submission candidates:', selected_names or 'none')
if len(selected_names) == 2:
    first = screen['evaluations'][selected_names[0]].out_of_fold_probabilities.to_numpy().argmax(axis=1)
    second = screen['evaluations'][selected_names[1]].out_of_fold_probabilities.to_numpy().argmax(axis=1)
    print(f"Selected-candidate OOF label disagreement: {np.mean(first != second):.3%}")
passing.style.format({
    'mean_accuracy': '{:.3%}', 'mean_gain': '{:+.3%}',
    'worst_fold_change': '{:+.3%}', 'repair_recall': '{:.3%}',
    'non_functional_recall': '{:.3%}', 'incumbent_disagreement': '{:.3%}',
    'current_leader_gain': '{:+.3%}', 'current_leader_disagreement': '{:.3%}',
})

Gate-passers: 79
Selected submission candidates: ['55% XGBoost depth 8 child 1 [current one-hot] + 45% Random Forest', '60% XGBoost depth 6/7/8 local bag + 40% Random Forest']
Selected-candidate OOF label disagreement: 1.172%


,mean_accuracy,mean_gain,fold_wins,worst_fold_change,repair_recall,non_functional_recall,incumbent_disagreement,current_leader_gain,current_leader_disagreement,passes_gate,beats_current_leader_gate
trial,,,,,,,,,,,
55% XGBoost depth 8 child 1 [current one-hot] + 45% Random Forest,81.625%,+0.223%,4,-0.021%,34.859%,78.433%,2.109%,+0.069%,1.115%,True,True
60% equal XGBoost depth 8 child 1 seed bag + 40% Random Forest,81.618%,+0.217%,4,-0.042%,34.714%,78.323%,2.178%,+0.063%,0.909%,True,True
60% second-wave XGBoost variant bag + 40% Random Forest,81.597%,+0.196%,4,+0.000%,34.309%,78.307%,2.058%,+0.042%,0.627%,True,False
60% XGBoost depth 6/7/8 local bag + 40% Random Forest,81.595%,+0.194%,5,+0.021%,34.396%,78.318%,2.159%,+0.040%,0.640%,True,False
65% XGBoost depth 8 child 1 [current one-hot] + 35% Random Forest,81.589%,+0.187%,4,-0.032%,34.193%,78.208%,2.487%,+0.034%,1.138%,True,False
65% equal XGBoost depth 8 child 1 seed bag + 35% Random Forest,81.587%,+0.185%,5,+0.042%,34.251%,78.235%,2.395%,+0.032%,1.016%,True,False
65% second-wave XGBoost variant bag + 35% Random Forest,81.582%,+0.181%,5,+0.011%,33.961%,78.203%,2.361%,+0.027%,0.882%,True,False
55% equal XGBoost depth 8 child 1 seed bag + 45% Random Forest,81.578%,+0.177%,4,-0.147%,34.830%,78.389%,2.052%,+0.023%,1.056%,True,False
55% XGBoost depth 6/7/8 local bag + 45% Random Forest,81.570%,+0.168%,4,-0.042%,34.338%,78.427%,1.972%,+0.015%,0.619%,True,False


In [6]:
report_path = RUNTIME_DIR / 'selected-submissions.json'
submission_records = json.loads(report_path.read_text(encoding='utf-8')) if report_path.exists() else []
public_scores = {
    '55% XGBoost depth 8 child 1 [current one-hot] + 45% Random Forest': 0.8241,
    '60% XGBoost depth 6/7/8 local bag + 40% Random Forest': 0.8240,
}
if submission_records:
    for record in submission_records:
        print(f"#{record['rank']} {record['candidate']}")
        print(f"  rows={record['rows']:,}; sha256={record['sha256']}")
        print(f"  class shares={record['competition_class_shares']}")
        public_score = public_scores.get(record['candidate'])
        print(f"  public leaderboard={public_score:.4f}" if public_score is not None else '  public leaderboard=not submitted')
else:
    print('No competition CSV was generated.')

#1 55% XGBoost depth 8 child 1 [current one-hot] + 45% Random Forest
  rows=14,850; sha256=53889121943054fa8ad8a5cea4e20b0ab357c151e9e88a9b4ee7a0898ba90e4a
  class shares={'functional': 0.6025589225589225, 'functional needs repair': 0.03952861952861953, 'non functional': 0.3579124579124579}
  public leaderboard=0.8241
#2 60% XGBoost depth 6/7/8 local bag + 40% Random Forest
  rows=14,850; sha256=cf87aabe4646cdddad58b7443e361e419c51e4f1f9d9a25269482ae4f4ce63e2
  class shares={'functional': 0.6054545454545455, 'functional needs repair': 0.03750841750841751, 'non functional': 0.35703703703703704}
  public leaderboard=0.8240
